# Experiment Tracking and Model Versioning

In this notebook, we extend the production machine learning workflow developed in the previous notebook by introducing **experiment tracking and model versioning using MLflow**.

The objective is to systematically track the experiments performed during uplift modeling and maintain important information such as:

- Model parameters and hyperparameters
- Evaluation metrics
- Dataset and experiment information
- Trained model artifacts
- Production predictions
- Experiment timestamps and run information

The **T-Learner** developed in Notebook 10 is used as the production baseline. MLflow provides a centralized way to record and organize these experiments, making the machine learning workflow more reproducible, auditable, and easier to manage.

This notebook focuses on creating an MLflow experiment, logging model parameters and evaluation metrics, and storing the trained model and prediction artifacts for future comparison and deployment.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime

import mlflow
import mlflow.sklearn

print("MLflow version:", mlflow.__version__)

C:\Users\ugand\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:151: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
C:\Users\ugand\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:186: UserWarning: Field name "schema" shadows an attribute in parent "BaseModel"; 
  warnings.warn(


MLflow version: 3.16.0


In [2]:
PROJECT_ROOT = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling"
)

DATA_PATH = (
    PROJECT_ROOT /
    "data" /
    "raw" /
    "criteo-research-uplift-v2.1.csv.gz"
)

PROCESSED_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed"
)

MODELS_DIR = (
    PROJECT_ROOT /
    "models"
)

REPORTS_DIR = (
    PROJECT_ROOT /
    "reports"
)

MLFLOW_DIR = (
    PROJECT_ROOT /
    "mlruns"
)

for directory in [
    PROCESSED_DIR,
    MODELS_DIR,
    REPORTS_DIR,
    MLFLOW_DIR
]:
    if not directory.exists():
        directory.mkdir(
            parents=True,
            exist_ok=True
        )

print("Project root:", PROJECT_ROOT)
print("MLflow directory:", MLFLOW_DIR)

Project root: C:\Users\ugand\customer-churn-uplift-modeling
MLflow directory: C:\Users\ugand\customer-churn-uplift-modeling\mlruns


In [3]:
mlflow_tracking_path = (
    MLFLOW_DIR / "mlflow.db"
)

mlflow.set_tracking_uri(
    f"sqlite:///{mlflow_tracking_path}"
)

print("MLflow tracking URI:")
print(mlflow.get_tracking_uri())

MLflow tracking URI:
sqlite:///C:\Users\ugand\customer-churn-uplift-modeling\mlruns\mlflow.db


In [4]:
EXPERIMENT_NAME = "customer_uplift_modeling"

mlflow.set_experiment(
    EXPERIMENT_NAME
)

print("Experiment:", EXPERIMENT_NAME)

2026/09/06 09:48:54 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/06 09:48:54 INFO mlflow.store.db.utils: Updating database tables
2026/09/06 09:48:57 INFO mlflow.tracking.fluent: Experiment with name 'customer_uplift_modeling' does not exist. Creating a new experiment.


Experiment: customer_uplift_modeling


In [5]:
metrics_path = (
    REPORTS_DIR /
    "production_model_metrics.csv"
)

production_metrics = pd.read_csv(
    metrics_path
)

display(production_metrics)

,metric,value
0,ROC-AUC,0.940856
1,PR-AUC,0.178952
2,Uplift@10%,0.149753
3,Uplift@20%,0.091474
4,Mean Predicted Uplift,0.013582


In [6]:
production_predictions_path = (
    PROCESSED_DIR /
    "production_uplift_predictions.csv"
)

production_predictions = pd.read_csv(
    production_predictions_path
)

print(
    "Prediction rows:",
    len(production_predictions)
)

display(
    production_predictions.head()
)

Prediction rows: 20000


,customer_index,actual_treatment,actual_conversion,predicted_control,predicted_treatment,predicted_uplift,rank,treatment_recommendation
0,7084,1,0,0.127235,0.713814,0.586579,1,Treat
1,10735,1,0,0.179967,0.761409,0.581442,2,Treat
2,4708,1,0,0.217890,0.791669,0.573779,3,Treat
3,54439,1,0,0.101021,0.662938,0.561917,4,Treat
4,49859,1,0,0.343001,0.903500,0.560499,5,Treat


In [7]:
metrics_dict = {}

for column in production_metrics.columns:
    value = production_metrics[column].iloc[0]

    if pd.notna(value):
        try:
            metrics_dict[column] = float(value)
        except (ValueError, TypeError):
            pass

print("Metrics to track:")

for key, value in metrics_dict.items():
    print(f"{key}: {value}")

Metrics to track:
value: 0.940856458756722


In [8]:
model_params = {
    "model_type": "T-Learner",
    "sample_size": 100_000,
    "test_size": 0.20,
    "random_state": 42,
    "n_estimators": 200,
    "max_depth": 10,
    "min_samples_leaf": 20,
    "class_weight": "balanced"
}

for key, value in model_params.items():
    print(f"{key}: {value}")

model_type: T-Learner
sample_size: 100000
test_size: 0.2
random_state: 42
n_estimators: 200
max_depth: 10
min_samples_leaf: 20
class_weight: balanced


In [9]:
with mlflow.start_run(
    run_name="t_learner_production_v1"
):

    # Log parameters
    mlflow.log_params(
        model_params
    )

    # Log metrics
    mlflow.log_metrics(
        metrics_dict
    )

    # Log dataset information
    mlflow.log_param(
        "dataset",
        "Criteo Uplift v2.1"
    )

    mlflow.log_param(
        "prediction_rows",
        len(production_predictions)
    )

    mlflow.log_param(
        "created_at",
        datetime.now().isoformat()
    )

    print(
        "MLflow run completed successfully."
    )

MLflow run completed successfully.


In [10]:
control_model_path = (
    MODELS_DIR /
    "t_learner_control_model.joblib"
)

treatment_model_path = (
    MODELS_DIR /
    "t_learner_treatment_model.joblib"
)

print("Control model:", control_model_path)
print("Treatment model:", treatment_model_path)

print(
    "\nControl exists:",
    control_model_path.exists()
)

print(
    "Treatment exists:",
    treatment_model_path.exists()
)

Control model: C:\Users\ugand\customer-churn-uplift-modeling\models\t_learner_control_model.joblib
Treatment model: C:\Users\ugand\customer-churn-uplift-modeling\models\t_learner_treatment_model.joblib

Control exists: True
Treatment exists: True


In [11]:
with mlflow.start_run(
    run_name="t_learner_production_v1_artifacts"
):

    mlflow.log_params(
        model_params
    )

    mlflow.log_metrics(
        metrics_dict
    )

    mlflow.log_artifact(
        str(control_model_path),
        artifact_path="models"
    )

    mlflow.log_artifact(
        str(treatment_model_path),
        artifact_path="models"
    )

    mlflow.log_artifact(
        str(metrics_path),
        artifact_path="metrics"
    )

    mlflow.log_artifact(
        str(production_predictions_path),
        artifact_path="predictions"
    )

    print(
        "Model artifacts logged successfully."
    )

Model artifacts logged successfully.


# Conclusion

In this notebook, experiment tracking and model artifact management were successfully integrated into the uplift modeling workflow using **MLflow**.

The main outcomes were:

- Created an MLflow experiment for the uplift modeling project.
- Configured a local MLflow tracking database within the project.
- Logged the T-Learner model parameters and hyperparameters.
- Recorded important model evaluation metrics such as ROC-AUC, PR-AUC, and uplift metrics.
- Tracked dataset and prediction information associated with the experiment.
- Logged the trained control and treatment models as MLflow artifacts.
- Stored production metrics and prediction outputs as tracked artifacts.
- Established a reproducible experiment record for the production T-Learner.

The workflow now provides a structured connection between **model development, evaluation, artifact storage, and experiment tracking**.

The next stage will focus on **model versioning and inspecting experiments through the MLflow interface**, moving the project closer to a production-oriented MLOps workflow.